In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import named_arrays as na
import msfc_ccd

In [ ]:
image = msfc_ccd.fits.open(msfc_ccd.samples.path_led_esis1)
taps = image.taps

axis_x = taps.axis_x
axis_y = taps.axis_y
axis_tap_x = taps.axis_tap_x
axis_tap_y = taps.axis_tap_y

sensor = taps.camera.sensor

In [ ]:
rows = {axis_y: slice(sensor.num_masked, None)}

signal = taps.unbiased.active.outputs[rows].mean(taps.axis_xy)

signal.ndarray

In [ ]:
columns = taps.unbiased.outputs[rows].mean(axis_y)

num_x = taps.num_x
num_last = num_x - sensor.num_overscan
index = np.arange(num_x - 12, num_x) - num_last

fig, ax = plt.subplots(constrained_layout=True)
for i in range(taps.shape[axis_tap_y]):
    for j in range(taps.shape[axis_tap_x]):
        tap = {axis_tap_y: i, axis_tap_x: j}
        ax.plot(
            index,
            columns[tap][{axis_x: slice(num_x - 12, None)}].ndarray.value,
            marker=".",
            label=taps.amplifier[tap].ndarray,
        )
ax.axvline(-0.5, color="gray", linestyle="--")
ax.set_yscale("log")
ax.set_xlabel("column, relative to the first overscan column")
ax.set_ylabel("mean signal (DN)")
ax.legend();

In [ ]:
cte = msfc_ccd.cte.eper(taps).outputs
cti = (1 - cte).to(u.dimensionless_unscaled)

for i in range(taps.shape[axis_tap_y]):
    for j in range(taps.shape[axis_tap_x]):
        tap = {axis_tap_y: i, axis_tap_x: j}
        print(
            f"{taps.amplifier[tap].ndarray}"
            f"   CTE {cte[tap].ndarray:0.6f}"
            f"   CTI {1e6 * cti[tap].ndarray.value:0.2f} x 10^-6"
        )